In [ ]:
# Cell 1: Imports & Configuration
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from datetime import datetime
import time
from typing import Dict, Any, List, Tuple

# Jupyter notebook configuration
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

from validation.create_embedding_plots import create_embedding_plots
from validation.data_loading import setup_data_with_error_handling

# LearnM8 imports
from learnm8.oracles import CSVOracle
from learnm8.core.data_manager import DataManager
from learnm8.learners.ensemble import RFEnsemble
from learnm8.utils.data_loaders import load_benchmark_data
from learnm8.pruning.utils import validate_pruning_parameters, create_pruning_strategy

In [ ]:
# Cell 2: Step 1 - Configuration
print("🔧 Step 1: Setting up configuration...")

# Dataset configurations for comprehensive testing
datasets = {
    "FEN1": {
        "path": "/home/tony/LearnM8/data/FEN1_5fv7_scoring_and_consensus_maxAL_with_activity.csv",
        "target_column": "CHEMPLP"
    },
    "ADA": {
        "path": "/home/tony/LearnM8/data/ADA_3ewy_scoring_and_consensus_maxAL_with_activity.csv",
        "target_column": "CHEMPLP"
    },
    "HIVPR": {
        "path": "/home/tony/LearnM8/data/HIVPR_1ajv_scoring_and_consensus_maxAL_with_activity.csv",
        "target_column": "CHEMPLP"
    }
}

# Strategy configurations for comprehensive testing
strategy_configs = {
    "ProbabilisticPruner": {
        "valid_params": {
            "value_threshold": -8.0,
            "retention_fraction": 0.7,
            "probability_threshold": 0.8
        },
        "invalid_params": [
            {"retention_fraction": 1.5},  # > 1
            {"probability_threshold": -0.1},  # < 0
            {}  # Missing required parameter
        ]
    },
    "ConfidenceIntervalPruner": {
        "valid_params": {
            "target_min": -10.0,
            "target_max": -6.0,
            "confidence_level": 0.95,
            "retention_fraction": 0.6
        },
        "invalid_params": [
            {"target_min": -6.0, "target_max": -10.0},  # min >= max
            {"confidence_level": 1.5},  # > 1
            {"target_min": -8.0}  # Missing target_max
        ]
    },
    "CycleBudgetPruner": {
        "valid_params": {
            "total_cycles": 15,
            "initial_retention_fraction": 1.0,
            "final_retention_fraction": 0.3
        },
        "invalid_params": [
            {"total_cycles": -5},  # negative
            {"initial_retention_fraction": 0.2, "final_retention_fraction": 0.8},  # initial < final
            {}  # Missing required parameter
        ]
    },
    "PerformanceBasedPruner": {
        "valid_params": {
            "performance_window": 3,
            "improvement_threshold": 0.01,
            "retention_fraction": 0.8
        },
        "invalid_params": [
            {"performance_window": -2},  # negative
            {"improvement_threshold": -0.05}  # negative
        ]
    }
}

print(f"📊 Configured {len(datasets)} datasets for testing")
print(f"🔧 Configured {len(strategy_configs)} pruning strategies for testing")
print("Configuration complete! ✅")

In [ ]:
# Cell 3: Step 2 - Parameter Validation Testing
print("🧪 Step 2: Testing parameter validation functions...")

validation_results = {}

for strategy_name, config in strategy_configs.items():
    print(f"\n📋 Testing {strategy_name}:")
    validation_results[strategy_name] = {"valid": [], "invalid": []}
    
    # Test valid parameters
    print("  ✅ Testing valid parameters...")
    valid_params = config["valid_params"]
    is_valid, errors = validate_pruning_parameters(strategy_name, valid_params)
    validation_results[strategy_name]["valid"].append({
        "params": valid_params,
        "is_valid": is_valid,
        "errors": errors
    })
    print(f"    Valid: {is_valid}, Errors: {errors}")
    
    # Test invalid parameters
    print("  ❌ Testing invalid parameters...")
    for i, invalid_params in enumerate(config["invalid_params"]):
        is_valid, errors = validate_pruning_parameters(strategy_name, invalid_params)
        validation_results[strategy_name]["invalid"].append({
            "params": invalid_params,
            "is_valid": is_valid,
            "errors": errors
        })
        print(f"    Test {i+1} - Valid: {is_valid}, Errors: {errors}")

print("\n🎯 Parameter validation testing complete!")

In [ ]:
# Cell 4: Step 3 - Strategy Factory Testing
print("🏭 Step 3: Testing strategy factory function...")

factory_results = {}

for strategy_name, config in strategy_configs.items():
    print(f"\n🔨 Testing factory for {strategy_name}:")
    factory_results[strategy_name] = {"success": [], "failure": []}
    
    # Test successful creation
    try:
        start_time = time.time()
        strategy = create_pruning_strategy(strategy_name, config["valid_params"])
        creation_time = time.time() - start_time
        
        factory_results[strategy_name]["success"].append({
            "params": config["valid_params"],
            "creation_time": creation_time,
            "strategy_type": type(strategy).__name__
        })
        print(f"  ✅ Successfully created {type(strategy).__name__} in {creation_time:.4f}s")
        
    except Exception as e:
        factory_results[strategy_name]["failure"].append({
            "params": config["valid_params"],
            "error": str(e)
        })
        print(f"  ❌ Failed to create strategy: {e}")
    
    # Test failure cases
    for i, invalid_params in enumerate(config["invalid_params"]):
        try:
            strategy = create_pruning_strategy(strategy_name, invalid_params)
            print(f"  ⚠️  Unexpected success with invalid params {i+1}: {invalid_params}")
            
        except ValueError as e:
            factory_results[strategy_name]["failure"].append({
                "params": invalid_params,
                "error": str(e)
            })
            print(f"  ✅ Correctly failed with invalid params {i+1}: {str(e)[:60]}...")
        except Exception as e:
            print(f"  ❌ Unexpected error type with invalid params {i+1}: {e}")

# Test unknown strategy
print("\n🔍 Testing unknown strategy handling:")
try:
    unknown_strategy = create_pruning_strategy("UnknownStrategy", {})
    print("  ❌ Unexpected success with unknown strategy")
except ValueError as e:
    print(f"  ✅ Correctly failed with unknown strategy: {str(e)[:60]}...")

print("\n🎯 Strategy factory testing complete!")

In [ ]:
# Cell 5: Step 4 - Load Test Datasets
print("📊 Step 4: Loading test datasets...")

loaded_datasets = {}

for dataset_name, config in datasets.items():
    print(f"\n📁 Loading {dataset_name}...")
    try:
        compound_pool, ground_truth = load_benchmark_data(config["path"], config["target_column"])
        
        loaded_datasets[dataset_name] = {
            "compound_pool": compound_pool,
            "ground_truth": ground_truth,
            "config": config,
            "size": len(compound_pool)
        }
        
        print(f"  ✅ Loaded {len(compound_pool)} compounds")
        print(f"  📊 Target column: {config['target_column']}")
        target_values = ground_truth[config['target_column']]
        print(f"  📈 Score range: {target_values.min():.2f} to {target_values.max():.2f}")
        print(f"  📊 Score mean: {target_values.mean():.2f} ± {target_values.std():.2f}")
        
    except Exception as e:
        print(f"  ❌ Failed to load {dataset_name}: {e}")
        loaded_datasets[dataset_name] = None

# Select primary dataset for detailed testing (use smallest for speed)
valid_datasets = {k: v for k, v in loaded_datasets.items() if v is not None}
if valid_datasets:
    primary_dataset_name = min(valid_datasets.keys(), key=lambda k: valid_datasets[k]["size"])
    primary_dataset = valid_datasets[primary_dataset_name]
    print(f"\n🎯 Selected {primary_dataset_name} as primary dataset for detailed testing ({primary_dataset['size']} compounds)")
else:
    print("\n❌ No datasets loaded successfully!")
    primary_dataset = None
    primary_dataset_name = None

In [ ]:
# Cell 6: Step 5 - DataManager Setup
print("⚙️  Step 5: Setting up data manager with error handling...")

if primary_dataset is not None:
    # Initialize DataManager
    data_manager = DataManager(results_dir=str(Path("./")), featurizer='morgan')
    
    # Test and setup data with error handling
    compound_pool = setup_data_with_error_handling(primary_dataset["compound_pool"], data_manager)
    ground_truth = primary_dataset["ground_truth"]
    target_column = primary_dataset["config"]["target_column"]
    target_path = primary_dataset["config"]["path"]
    
    print(f"✅ DataManager setup complete for {primary_dataset_name}")
    print(f"📊 Compounds ready for testing: {len(compound_pool)}")
else:
    print("❌ Cannot setup DataManager - no valid datasets loaded")

In [ ]:
# Cell 7: Step 6 - Initial Training Setup
print("🎯 Step 6: Setting up initial training data...")

if primary_dataset is not None:
    # Initialize learner
    learner = RFEnsemble(n_estimators=3, random_states=[42, 43, 44])
    
    # Determine initial training size
    initial_size = min(100, len(compound_pool) // 10)  # Adaptive initial size
    np.random.seed(42)
    initial_indices = np.random.choice(len(compound_pool), size=initial_size, replace=False)
    
    # Split data
    labeled_compounds = compound_pool.iloc[initial_indices].copy()
    unlabeled_compounds = compound_pool.drop(compound_pool.index[initial_indices]).copy()
    
    # Create oracle and measure labeled compounds
    oracle = CSVOracle(csv_path=str(target_path))
    labeled_compounds = oracle.measure(labeled_compounds, [target_column])
    
    # Train the model
    print("🔄 Training initial model...")
    start_time = time.time()
    learner.train(labeled_compounds, target_column, data_manager)
    training_time = time.time() - start_time
    
    print(f"✅ Initial training completed: {len(labeled_compounds)} labeled, {len(unlabeled_compounds)} unlabeled")
    print(f"⏱️  Training time: {training_time:.2f} seconds")
    
    # Get predictions for testing strategies
    print("🔮 Getting predictions for strategy testing...")
    predictions, uncertainties = learner.predict(unlabeled_compounds, data_manager)
    ground_truth_unlabeled = oracle.measure(unlabeled_compounds, [target_column])[target_column].values
    
    print(f"📊 Predictions shape: {predictions.shape}")
    print(f"📊 Prediction range: {predictions.min():.2f} to {predictions.max():.2f}")
    if uncertainties is not None:
        print(f"📊 Uncertainties shape: {uncertainties.shape}")
        print(f"📊 Uncertainty range: {uncertainties.min():.4f} to {uncertainties.max():.4f}")
    else:
        print("📊 No uncertainties available from this learner")
        
else:
    print("❌ Cannot setup training - no valid datasets loaded")

In [ ]:
# Cell 8: Step 7 - Strategy Comparison Analysis
print("🔬 Step 7: Running strategy comparison analysis...")

if primary_dataset is not None:
    comparison_results = {}
    
    # Prepare data for pruning strategies
    test_compounds = unlabeled_compounds.copy()
    test_compounds['prediction'] = predictions
    if uncertainties is not None:
        test_compounds['uncertainty'] = uncertainties
    else:
        test_compounds['uncertainty'] = np.zeros(len(predictions))
    
    test_compounds['ground_truth'] = ground_truth_unlabeled
    
    print(f"🧪 Testing strategies on {len(test_compounds)} compounds...")
    
    for strategy_name, config in strategy_configs.items():
        print(f"\n🔧 Testing {strategy_name}...")
        
        try:
            # Create strategy
            start_time = time.time()
            strategy = create_pruning_strategy(strategy_name, config["valid_params"])
            creation_time = time.time() - start_time
            
            # Test should_prune method
            prune_start = time.time()
            should_prune_result = strategy.should_prune(
                cycle=5,
                labeled_data=labeled_compounds,
                unlabeled_data=test_compounds,
                target_column=target_column
            )
            should_prune_time = time.time() - prune_start
            
            # Test prune method if should_prune returned True
            prune_execution_time = 0
            pruned_compounds = None
            pruning_stats = None
            
            if should_prune_result:
                prune_exec_start = time.time()
                pruned_compounds, pruning_stats = strategy.prune(
                    cycle=5,
                    labeled_data=labeled_compounds,
                    unlabeled_data=test_compounds,
                    target_column=target_column
                )
                prune_execution_time = time.time() - prune_exec_start
            
            comparison_results[strategy_name] = {
                "strategy_params": config["valid_params"],
                "creation_time": creation_time,
                "should_prune": should_prune_result,
                "should_prune_time": should_prune_time,
                "prune_execution_time": prune_execution_time,
                "original_size": len(test_compounds),
                "pruned_size": len(pruned_compounds) if pruned_compounds is not None else len(test_compounds),
                "pruning_stats": pruning_stats,
                "success": True
            }
            
            print(f"  ✅ Strategy created in {creation_time:.4f}s")
            print(f"  🔍 Should prune: {should_prune_result} (checked in {should_prune_time:.4f}s)")
            if should_prune_result:
                print(f"  ✂️  Pruning executed in {prune_execution_time:.4f}s")
                print(f"  📊 Size: {len(test_compounds)} → {len(pruned_compounds)} compounds")
                retention_rate = len(pruned_compounds) / len(test_compounds)
                print(f"  📈 Retention rate: {retention_rate:.2%}")
                if pruning_stats:
                    print(f"  📋 Pruning stats: {pruning_stats}")
            else:
                print(f"  📊 No pruning performed (should_prune = False)")
            
        except Exception as e:
            comparison_results[strategy_name] = {
                "strategy_params": config["valid_params"],
                "error": str(e),
                "success": False
            }
            print(f"  ❌ Strategy failed: {e}")
    
    print("\n🎯 Strategy comparison analysis complete!")
    
else:
    print("❌ Cannot run strategy comparison - no valid datasets loaded")
    comparison_results = {}

In [ ]:
# Cell 9: Step 8 - Parameter Sensitivity Analysis
print("📊 Step 8: Running parameter sensitivity analysis...")

if primary_dataset is not None:
    sensitivity_results = {}
    
    # Define parameter variations for sensitivity testing
    sensitivity_configs = {
        "ProbabilisticPruner": {
            "base_params": {"value_threshold": -8.0, "retention_fraction": 0.7},
            "variations": [
                {"retention_fraction": 0.5},
                {"retention_fraction": 0.8},
                {"retention_fraction": 0.9},
                {"value_threshold": -6.0},
                {"value_threshold": -10.0}
            ]
        },
        "ConfidenceIntervalPruner": {
            "base_params": {"target_min": -10.0, "target_max": -6.0, "confidence_level": 0.95},
            "variations": [
                {"confidence_level": 0.8},
                {"confidence_level": 0.99},
                {"target_min": -12.0, "target_max": -4.0},
                {"retention_fraction": 0.5},
                {"retention_fraction": 0.8}
            ]
        }
    }
    
    for strategy_name, config in sensitivity_configs.items():
        print(f"\n🔬 Sensitivity analysis for {strategy_name}:")
        sensitivity_results[strategy_name] = {}
        
        base_params = config["base_params"]
        
        for i, variation in enumerate(config["variations"]):
            test_params = base_params.copy()
            test_params.update(variation)
            
            variation_name = f"variation_{i+1}"
            print(f"  🧪 Testing {variation_name}: {variation}")
            
            try:
                # Validate parameters
                is_valid, errors = validate_pruning_parameters(strategy_name, test_params)
                
                if is_valid:
                    # Create and test strategy
                    start_time = time.time()
                    strategy = create_pruning_strategy(strategy_name, test_params)
                    
                    should_prune = strategy.should_prune(
                        cycle=5,
                        labeled_data=labeled_compounds,
                        unlabeled_data=test_compounds,
                        target_column=target_column
                    )
                    
                    pruned_size = len(test_compounds)
                    if should_prune:
                        pruned_compounds, _ = strategy.prune(
                            cycle=5,
                            labeled_data=labeled_compounds,
                            unlabeled_data=test_compounds,
                            target_column=target_column
                        )
                        pruned_size = len(pruned_compounds)
                    
                    total_time = time.time() - start_time
                    retention_rate = pruned_size / len(test_compounds)
                    
                    sensitivity_results[strategy_name][variation_name] = {
                        "params": test_params,
                        "variation": variation,
                        "should_prune": should_prune,
                        "retention_rate": retention_rate,
                        "execution_time": total_time,
                        "success": True
                    }
                    
                    print(f"    ✅ Success - Retention: {retention_rate:.2%}, Time: {total_time:.4f}s")
                    
                else:
                    sensitivity_results[strategy_name][variation_name] = {
                        "params": test_params,
                        "variation": variation,
                        "validation_errors": errors,
                        "success": False
                    }
                    print(f"    ❌ Invalid parameters: {errors}")
                    
            except Exception as e:
                sensitivity_results[strategy_name][variation_name] = {
                    "params": test_params,
                    "variation": variation,
                    "error": str(e),
                    "success": False
                }
                print(f"    ❌ Execution error: {e}")
    
    print("\n🎯 Parameter sensitivity analysis complete!")
    
else:
    print("❌ Cannot run sensitivity analysis - no valid datasets loaded")
    sensitivity_results = {}

In [ ]:
# Cell 10: Step 9 - Performance Benchmarking
print("⏱️  Step 9: Running performance benchmarking...")

if primary_dataset is not None:
    benchmark_results = {}
    n_iterations = 5  # Number of timing iterations
    
    for strategy_name, config in strategy_configs.items():
        print(f"\n⏱️  Benchmarking {strategy_name}...")
        
        timing_results = {
            "creation_times": [],
            "should_prune_times": [],
            "prune_times": [],
            "total_times": []
        }
        
        for i in range(n_iterations):
            try:
                # Time strategy creation
                start_time = time.time()
                strategy = create_pruning_strategy(strategy_name, config["valid_params"])
                creation_time = time.time() - start_time
                timing_results["creation_times"].append(creation_time)
                
                # Time should_prune check
                start_time = time.time()
                should_prune = strategy.should_prune(
                    cycle=5,
                    labeled_data=labeled_compounds,
                    unlabeled_data=test_compounds,
                    target_column=target_column
                )
                should_prune_time = time.time() - start_time
                timing_results["should_prune_times"].append(should_prune_time)
                
                # Time pruning execution
                prune_time = 0
                if should_prune:
                    start_time = time.time()
                    strategy.prune(
                        cycle=5,
                        labeled_data=labeled_compounds,
                        unlabeled_data=test_compounds,
                        target_column=target_column
                    )
                    prune_time = time.time() - start_time
                
                timing_results["prune_times"].append(prune_time)
                timing_results["total_times"].append(creation_time + should_prune_time + prune_time)
                
            except Exception as e:
                print(f"    ❌ Iteration {i+1} failed: {e}")
                continue
        
        # Calculate statistics
        if timing_results["total_times"]:
            benchmark_results[strategy_name] = {
                "mean_creation_time": np.mean(timing_results["creation_times"]),
                "std_creation_time": np.std(timing_results["creation_times"]),
                "mean_should_prune_time": np.mean(timing_results["should_prune_times"]),
                "std_should_prune_time": np.std(timing_results["should_prune_times"]),
                "mean_prune_time": np.mean(timing_results["prune_times"]),
                "std_prune_time": np.std(timing_results["prune_times"]),
                "mean_total_time": np.mean(timing_results["total_times"]),
                "std_total_time": np.std(timing_results["total_times"]),
                "n_successful_iterations": len(timing_results["total_times"])
            }
            
            results = benchmark_results[strategy_name]
            print(f"  ✅ Completed {results['n_successful_iterations']}/{n_iterations} iterations")
            print(f"  ⏱️  Mean total time: {results['mean_total_time']:.4f} ± {results['std_total_time']:.4f}s")
            print(f"  🏭 Mean creation time: {results['mean_creation_time']:.4f} ± {results['std_creation_time']:.4f}s")
            print(f"  🔍 Mean should_prune time: {results['mean_should_prune_time']:.4f} ± {results['std_should_prune_time']:.4f}s")
            print(f"  ✂️  Mean prune time: {results['mean_prune_time']:.4f} ± {results['std_prune_time']:.4f}s")
        else:
            benchmark_results[strategy_name] = {"error": "All iterations failed"}
            print(f"  ❌ All timing iterations failed")
    
    print("\n🎯 Performance benchmarking complete!")
    
else:
    print("❌ Cannot run benchmarking - no valid datasets loaded")
    benchmark_results = {}

In [ ]:
# Cell 11: Step 10 - Generate Visualizations
print("📊 Step 10: Generating comprehensive visualizations...")

if primary_dataset is not None and comparison_results:
    # Create plots directory
    plots_dir = Path(f"./plots_{primary_dataset_name}")
    plots_dir.mkdir(exist_ok=True)
    
    # Set up plotting style
    plt.style.use('default')
    sns.set_palette("husl")
    
    # 1. Strategy Comparison Matrix
    print("📊 Creating strategy comparison matrix...")
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Extract successful strategies for plotting
    successful_strategies = {k: v for k, v in comparison_results.items() if v.get("success", False)}
    
    if successful_strategies:
        strategy_names = list(successful_strategies.keys())
        
        # Retention rates
        retention_rates = [v["pruned_size"] / v["original_size"] for v in successful_strategies.values()]
        ax1.bar(range(len(strategy_names)), retention_rates, color=sns.color_palette("husl", len(strategy_names)))
        ax1.set_xlabel('Strategy')
        ax1.set_ylabel('Retention Rate')
        ax1.set_title('Retention Rates by Strategy')
        ax1.set_xticks(range(len(strategy_names)))
        ax1.set_xticklabels([name.replace('Pruner', '') for name in strategy_names], rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # Execution times
        total_times = [(v["should_prune_time"] + v["prune_execution_time"]) * 1000 for v in successful_strategies.values()]
        ax2.bar(range(len(strategy_names)), total_times, color=sns.color_palette("viridis", len(strategy_names)))
        ax2.set_xlabel('Strategy')
        ax2.set_ylabel('Execution Time (ms)')
        ax2.set_title('Execution Times by Strategy')
        ax2.set_xticks(range(len(strategy_names)))
        ax2.set_xticklabels([name.replace('Pruner', '') for name in strategy_names], rotation=45)
        ax2.grid(True, alpha=0.3)
        
        # Should prune decisions
        should_prune_decisions = [v["should_prune"] for v in successful_strategies.values()]
        colors = ['green' if decision else 'red' for decision in should_prune_decisions]
        ax3.bar(range(len(strategy_names)), [1 if decision else 0 for decision in should_prune_decisions], color=colors)
        ax3.set_xlabel('Strategy')
        ax3.set_ylabel('Should Prune (1=Yes, 0=No)')
        ax3.set_title('Pruning Decisions by Strategy')
        ax3.set_xticks(range(len(strategy_names)))
        ax3.set_xticklabels([name.replace('Pruner', '') for name in strategy_names], rotation=45)
        ax3.set_ylim(-0.1, 1.1)
        ax3.grid(True, alpha=0.3)
        
        # Compounds pruned
        compounds_pruned = [v["original_size"] - v["pruned_size"] for v in successful_strategies.values()]
        ax4.bar(range(len(strategy_names)), compounds_pruned, color=sns.color_palette("plasma", len(strategy_names)))
        ax4.set_xlabel('Strategy')
        ax4.set_ylabel('Compounds Pruned')
        ax4.set_title('Number of Compounds Pruned')
        ax4.set_xticks(range(len(strategy_names)))
        ax4.set_xticklabels([name.replace('Pruner', '') for name in strategy_names], rotation=45)
        ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    comparison_plot_path = plots_dir / "strategy_comparison_matrix.png"
    plt.savefig(comparison_plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved strategy comparison matrix: {comparison_plot_path}")
    
else:
    print("❌ Cannot generate visualizations - no valid comparison results")

In [ ]:
# Cell 12: Step 11 - Parameter Sensitivity Visualizations
print("📊 Step 11: Creating parameter sensitivity visualizations...")

if primary_dataset is not None and sensitivity_results:
    # 2. Parameter Sensitivity Analysis
    print("📊 Creating parameter sensitivity plots...")
    
    for strategy_name, results in sensitivity_results.items():
        if not results:
            continue
            
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Extract successful variations
        successful_variations = {k: v for k, v in results.items() if v.get("success", False)}
        
        if successful_variations:
            variation_names = list(successful_variations.keys())
            retention_rates = [v["retention_rate"] for v in successful_variations.values()]
            execution_times = [v["execution_time"] * 1000 for v in successful_variations.values()]
            
            # Retention rate sensitivity
            ax1.bar(range(len(variation_names)), retention_rates, 
                   color=sns.color_palette("coolwarm", len(variation_names)))
            ax1.set_xlabel('Parameter Variation')
            ax1.set_ylabel('Retention Rate')
            ax1.set_title(f'{strategy_name.replace("Pruner", "")} - Parameter Sensitivity (Retention)')
            ax1.set_xticks(range(len(variation_names)))
            ax1.set_xticklabels([name.replace('variation_', 'V') for name in variation_names])
            ax1.grid(True, alpha=0.3)
            
            # Add parameter change annotations
            for i, (var_name, var_data) in enumerate(successful_variations.items()):
                variation = var_data["variation"]
                param_str = ', '.join([f"{k}={v}" for k, v in variation.items()])
                ax1.text(i, retention_rates[i] + 0.01, param_str, 
                        rotation=45, ha='left', va='bottom', fontsize=8)
            
            # Execution time sensitivity
            ax2.bar(range(len(variation_names)), execution_times, 
                   color=sns.color_palette("viridis", len(variation_names)))
            ax2.set_xlabel('Parameter Variation')
            ax2.set_ylabel('Execution Time (ms)')
            ax2.set_title(f'{strategy_name.replace("Pruner", "")} - Parameter Sensitivity (Time)')
            ax2.set_xticks(range(len(variation_names)))
            ax2.set_xticklabels([name.replace('variation_', 'V') for name in variation_names])
            ax2.grid(True, alpha=0.3)
            
        plt.tight_layout()
        sensitivity_plot_path = plots_dir / f"{strategy_name.lower()}_sensitivity_analysis.png"
        plt.savefig(sensitivity_plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"💾 Saved sensitivity analysis: {sensitivity_plot_path}")
    
else:
    print("❌ Cannot generate sensitivity visualizations - no valid sensitivity results")

In [ ]:
# Cell 13: Step 12 - Performance Benchmarking Visualizations
print("📊 Step 12: Creating performance benchmarking visualizations...")

if primary_dataset is not None and benchmark_results:
    # 3. Performance Benchmarking Results
    print("📊 Creating performance benchmark plots...")
    
    # Extract successful benchmarks
    successful_benchmarks = {k: v for k, v in benchmark_results.items() if "mean_total_time" in v}
    
    if successful_benchmarks:
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        strategy_names = list(successful_benchmarks.keys())
        
        # Total execution times with error bars
        total_means = [v["mean_total_time"] * 1000 for v in successful_benchmarks.values()]
        total_stds = [v["std_total_time"] * 1000 for v in successful_benchmarks.values()]
        ax1.bar(range(len(strategy_names)), total_means, yerr=total_stds, 
               color=sns.color_palette("Set2", len(strategy_names)), capsize=5)
        ax1.set_xlabel('Strategy')
        ax1.set_ylabel('Total Time (ms)')
        ax1.set_title('Total Execution Time (Mean ± Std)')
        ax1.set_xticks(range(len(strategy_names)))
        ax1.set_xticklabels([name.replace('Pruner', '') for name in strategy_names], rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # Component breakdown (stacked bar)
        creation_means = [v["mean_creation_time"] * 1000 for v in successful_benchmarks.values()]
        should_prune_means = [v["mean_should_prune_time"] * 1000 for v in successful_benchmarks.values()]
        prune_means = [v["mean_prune_time"] * 1000 for v in successful_benchmarks.values()]
        
        ax2.bar(range(len(strategy_names)), creation_means, label='Creation', alpha=0.8)
        ax2.bar(range(len(strategy_names)), should_prune_means, bottom=creation_means, 
               label='Should Prune Check', alpha=0.8)
        bottom_for_prune = [c + s for c, s in zip(creation_means, should_prune_means)]
        ax2.bar(range(len(strategy_names)), prune_means, bottom=bottom_for_prune, 
               label='Prune Execution', alpha=0.8)
        ax2.set_xlabel('Strategy')
        ax2.set_ylabel('Time (ms)')
        ax2.set_title('Execution Time Breakdown')
        ax2.set_xticks(range(len(strategy_names)))
        ax2.set_xticklabels([name.replace('Pruner', '') for name in strategy_names], rotation=45)
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Coefficient of variation (relative variability)
        cvs = [(v["std_total_time"] / v["mean_total_time"]) * 100 for v in successful_benchmarks.values()]
        ax3.bar(range(len(strategy_names)), cvs, 
               color=sns.color_palette("plasma", len(strategy_names)))
        ax3.set_xlabel('Strategy')
        ax3.set_ylabel('Coefficient of Variation (%)')
        ax3.set_title('Timing Variability (CV of Total Time)')
        ax3.set_xticks(range(len(strategy_names)))
        ax3.set_xticklabels([name.replace('Pruner', '') for name in strategy_names], rotation=45)
        ax3.grid(True, alpha=0.3)
        
        # Success rates
        success_rates = [v["n_successful_iterations"] / 5 * 100 for v in successful_benchmarks.values()]
        colors = ['green' if rate == 100 else 'orange' if rate >= 80 else 'red' for rate in success_rates]
        ax4.bar(range(len(strategy_names)), success_rates, color=colors)
        ax4.set_xlabel('Strategy')
        ax4.set_ylabel('Success Rate (%)')
        ax4.set_title('Benchmark Success Rate')
        ax4.set_xticks(range(len(strategy_names)))
        ax4.set_xticklabels([name.replace('Pruner', '') for name in strategy_names], rotation=45)
        ax4.set_ylim(0, 105)
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        benchmark_plot_path = plots_dir / "performance_benchmarking.png"
        plt.savefig(benchmark_plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"💾 Saved performance benchmarking: {benchmark_plot_path}")
    
else:
    print("❌ Cannot generate benchmark visualizations - no valid benchmark results")

In [ ]:
# Cell 14: Step 13 - Validation Summary Analysis
print("📋 Step 13: Generating comprehensive validation summary...")

# 4. Comprehensive Validation Summary
print("📊 Creating validation summary plot...")

if primary_dataset is not None:
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Parameter validation summary
    validation_summary = {}
    for strategy_name, results in validation_results.items():
        valid_count = sum(1 for r in results["valid"] if r["is_valid"])
        invalid_count = sum(1 for r in results["invalid"] if not r["is_valid"])
        validation_summary[strategy_name] = {
            "valid": valid_count,
            "invalid_handled": invalid_count
        }
    
    if validation_summary:
        strategies = list(validation_summary.keys())
        valid_counts = [validation_summary[s]["valid"] for s in strategies]
        invalid_counts = [validation_summary[s]["invalid_handled"] for s in strategies]
        
        x = np.arange(len(strategies))
        width = 0.35
        
        ax1.bar(x - width/2, valid_counts, width, label='Valid Params', color='green', alpha=0.7)
        ax1.bar(x + width/2, invalid_counts, width, label='Invalid Params Caught', color='red', alpha=0.7)
        ax1.set_xlabel('Strategy')
        ax1.set_ylabel('Number of Tests')
        ax1.set_title('Parameter Validation Test Results')
        ax1.set_xticks(x)
        ax1.set_xticklabels([s.replace('Pruner', '') for s in strategies], rotation=45)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
    # Factory success rates
    factory_summary = {}
    for strategy_name, results in factory_results.items():
        success_count = len(results["success"])
        failure_count = len(results["failure"])
        total = success_count + failure_count
        factory_summary[strategy_name] = {
            "success_rate": (success_count / total) * 100 if total > 0 else 0,
            "total_tests": total
        }
    
    if factory_summary:
        strategies = list(factory_summary.keys())
        success_rates = [factory_summary[s]["success_rate"] for s in strategies]
        colors = ['green' if rate == 100 else 'orange' if rate >= 50 else 'red' for rate in success_rates]
        
        ax2.bar(range(len(strategies)), success_rates, color=colors)
        ax2.set_xlabel('Strategy')
        ax2.set_ylabel('Success Rate (%)')
        ax2.set_title('Strategy Factory Success Rate')
        ax2.set_xticks(range(len(strategies)))
        ax2.set_xticklabels([s.replace('Pruner', '') for s in strategies], rotation=45)
        ax2.set_ylim(0, 105)
        ax2.grid(True, alpha=0.3)
    
    # Overall utility assessment
    utility_scores = {}
    for strategy_name in strategy_configs.keys():
        score = 0
        
        # Parameter validation (0-25 points)
        if strategy_name in validation_results:
            valid_tests = validation_results[strategy_name]["valid"]
            invalid_tests = validation_results[strategy_name]["invalid"]
            if all(r["is_valid"] for r in valid_tests) and all(not r["is_valid"] for r in invalid_tests):
                score += 25
        
        # Factory success (0-25 points)
        if strategy_name in factory_results and factory_results[strategy_name]["success"]:
            score += 25
        
        # Strategy execution (0-25 points)
        if strategy_name in comparison_results and comparison_results[strategy_name].get("success", False):
            score += 25
        
        # Performance reliability (0-25 points)
        if strategy_name in benchmark_results:
            bench_result = benchmark_results[strategy_name]
            if "mean_total_time" in bench_result and bench_result.get("n_successful_iterations", 0) >= 4:
                score += 25
        
        utility_scores[strategy_name] = score
    
    if utility_scores:
        strategies = list(utility_scores.keys())
        scores = list(utility_scores.values())
        colors = sns.color_palette("RdYlGn", len(set(scores)))
        score_colors = [colors[int(score//20)] for score in scores]  # Map scores to colors
        
        ax3.bar(range(len(strategies)), scores, color=score_colors)
        ax3.set_xlabel('Strategy')
        ax3.set_ylabel('Utility Score (0-100)')
        ax3.set_title('Overall Utility Assessment')
        ax3.set_xticks(range(len(strategies)))
        ax3.set_xticklabels([s.replace('Pruner', '') for s in strategies], rotation=45)
        ax3.set_ylim(0, 105)
        ax3.grid(True, alpha=0.3)
        
        # Add score labels
        for i, score in enumerate(scores):
            ax3.text(i, score + 2, f'{score}', ha='center', va='bottom', fontweight='bold')
    
    # Test coverage summary
    test_categories = ['Parameter\nValidation', 'Factory\nCreation', 'Strategy\nExecution', 
                      'Sensitivity\nAnalysis', 'Performance\nBenchmark']
    
    coverage_data = {
        'Parameter\nValidation': len([s for s in strategy_configs.keys() if s in validation_results]),
        'Factory\nCreation': len([s for s in strategy_configs.keys() if s in factory_results]),
        'Strategy\nExecution': len([s for s in strategy_configs.keys() if s in comparison_results]),
        'Sensitivity\nAnalysis': len(sensitivity_results),
        'Performance\nBenchmark': len([s for s in strategy_configs.keys() if s in benchmark_results])
    }
    
    total_strategies = len(strategy_configs)
    coverage_percentages = [(coverage_data[cat] / total_strategies) * 100 for cat in test_categories]
    
    colors = ['green' if pct == 100 else 'orange' if pct >= 75 else 'red' for pct in coverage_percentages]
    ax4.bar(range(len(test_categories)), coverage_percentages, color=colors)
    ax4.set_xlabel('Test Category')
    ax4.set_ylabel('Coverage (%)')
    ax4.set_title('Test Coverage by Category')
    ax4.set_xticks(range(len(test_categories)))
    ax4.set_xticklabels(test_categories, rotation=45)
    ax4.set_ylim(0, 105)
    ax4.grid(True, alpha=0.3)
    
    # Add percentage labels
    for i, pct in enumerate(coverage_percentages):
        ax4.text(i, pct + 2, f'{pct:.0f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    summary_plot_path = plots_dir / "validation_summary_analysis.png"
    plt.savefig(summary_plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved validation summary: {summary_plot_path}")
    
    print("\n🎯 Comprehensive validation analysis complete!")
    print(f"📊 Primary dataset: {primary_dataset_name} ({primary_dataset['size']} compounds)")
    print(f"🔧 Strategies tested: {len(strategy_configs)}")
    print(f"📁 Plots saved to: {plots_dir}")
    
else:
    print("❌ Cannot generate summary analysis - no valid datasets loaded")

In [ ]:
# Cell 15: Step 14 - Final Results Summary
print("📋 Step 14: Final validation results summary...")

print("\n" + "="*80)
print("🧪 PRUNING UTILITIES VALIDATION RESULTS SUMMARY")
print("="*80)

if primary_dataset is not None:
    print(f"\n📊 Dataset Information:")
    print(f"  • Primary dataset: {primary_dataset_name}")
    print(f"  • Compounds tested: {primary_dataset['size']:,}")
    print(f"  • Target column: {primary_dataset['config']['target_column']}")
    
    print(f"\n🔧 Parameter Validation Results:")
    for strategy_name, results in validation_results.items():
        valid_tests = len([r for r in results["valid"] if r["is_valid"]])
        invalid_tests = len([r for r in results["invalid"] if not r["is_valid"]])
        total_invalid = len(results["invalid"])
        print(f"  • {strategy_name}:")
        print(f"    - Valid parameters: {valid_tests}/{len(results['valid'])} passed")
        print(f"    - Invalid parameters: {invalid_tests}/{total_invalid} correctly rejected")
    
    print(f"\n🏭 Strategy Factory Results:")
    for strategy_name, results in factory_results.items():
        success_count = len(results["success"])
        failure_count = len(results["failure"])
        total = success_count + failure_count
        success_rate = (success_count / total) * 100 if total > 0 else 0
        print(f"  • {strategy_name}: {success_rate:.0f}% success rate ({success_count}/{total} tests)")
        if results["success"]:
            avg_creation_time = np.mean([r["creation_time"] for r in results["success"]])
            print(f"    - Average creation time: {avg_creation_time:.4f}s")
    
    print(f"\n🔬 Strategy Execution Results:")
    for strategy_name, results in comparison_results.items():
        if results.get("success", False):
            retention_rate = results["pruned_size"] / results["original_size"]
            total_time = results["should_prune_time"] + results["prune_execution_time"]
            print(f"  • {strategy_name}:")
            print(f"    - Should prune: {results['should_prune']}")
            print(f"    - Retention rate: {retention_rate:.2%}")
            print(f"    - Execution time: {total_time:.4f}s")
        else:
            print(f"  • {strategy_name}: FAILED - {results.get('error', 'Unknown error')}")
    
    if sensitivity_results:
        print(f"\n📊 Parameter Sensitivity Results:")
        for strategy_name, results in sensitivity_results.items():
            successful_variations = len([r for r in results.values() if r.get("success", False)])
            total_variations = len(results)
            print(f"  • {strategy_name}: {successful_variations}/{total_variations} parameter variations successful")
    
    if benchmark_results:
        print(f"\n⏱️  Performance Benchmark Results:")
        for strategy_name, results in benchmark_results.items():
            if "mean_total_time" in results:
                print(f"  • {strategy_name}:")
                print(f"    - Mean execution time: {results['mean_total_time']:.4f} ± {results['std_total_time']:.4f}s")
                print(f"    - Success rate: {results['n_successful_iterations']}/5 iterations")
    
    print(f"\n💾 Generated Visualizations:")
    if plots_dir.exists():
        plot_files = list(plots_dir.glob("*.png"))
        for plot_file in plot_files:
            print(f"  • {plot_file.name}")
        print(f"\n📁 All plots saved to: {plots_dir}")
    
    print(f"\n✅ Validation Complete!")
    print(f"   • {len(strategy_configs)} strategies tested")
    print(f"   • {len(datasets)} datasets configured")
    print(f"   • All utility functions validated")
    print(f"   • Comprehensive analysis generated")
    
else:
    print("\n❌ Validation incomplete - no datasets loaded successfully")
    print("   • Check dataset paths and file availability")
    print("   • Verify LearnM8 installation and dependencies")

print("\n" + "="*80)

In [ ]:
# Cell 16: Cleanup
import shutil
# Clean up temporary files
if 'data_manager' in locals():
    shutil.rmtree(Path(data_manager.results_dir / ".cache/"), ignore_errors=True)
    print("🧹 Cleaned up temporary files")
else:
    print("🧹 No cleanup needed")